In [ ]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import animation
from IPython.display import HTML, display
import tqdm as tqdm
import os

In [ ]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/bronze/manifests/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/bronze/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

def metadata_path_for_video(video_path):
    video_stem = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(
        "/home/guilherme_monteiro/projetos/tcc/data/silver/face_metadata_json",
        f"{video_stem}_meta.json"
    )

df['metadata_path'] = df['video_path'].apply(metadata_path_for_video)
df['has_metadata'] = df['metadata_path'].apply(os.path.exists)
df_meta = df[df['has_metadata']].reset_index(drop=True)

print(f"Vídeos no CSV: {len(df)}")
print(f"Vídeos com metadata disponível: {len(df_meta)}")
print(df_meta['Video Ground Truth'].value_counts())

# Funcoes basicas

In [ ]:
def get_video_frame_count(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frame_count


def sample_frame_indices(frame_count, max_frames=None):
    if frame_count <= 0:
        return np.array([], dtype=int)

    if max_frames is None or frame_count <= max_frames:
        return np.arange(frame_count, dtype=int)

    return np.linspace(0, frame_count - 1, int(max_frames)).astype(int)


def load_video_frames(video_path, max_frames=None, return_indices=False):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = sample_frame_indices(frame_count, max_frames)

    frames = []
    used_indices = []

    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()

        if not ret:
            continue

        frames.append(frame)
        used_indices.append(int(frame_idx))

    cap.release()

    frames = np.array(frames)

    if return_indices:
        return frames, np.array(used_indices, dtype=int), frame_count

    return frames


def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]
    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)
    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return frame_resized, scale


def load_metadata(video_path):
    full_path = metadata_path_for_video(video_path)
    with open(full_path, "r") as f:
        return json.load(f)


def metadata_index_for_frame(frame_idx, frame_count, metadata_len):
    if metadata_len <= 0:
        return None

    if metadata_len >= frame_count - 3:
        return min(int(frame_idx), metadata_len - 1)

    metadata_frame_indices = sample_frame_indices(frame_count, metadata_len)
    return int(np.argmin(np.abs(metadata_frame_indices - int(frame_idx))))


def scale_bbox(bbox, scale):
    return [int(round(x * scale)) for x in bbox]


def clip_bbox(bbox, width, height, min_size=4):
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]

    x1 = max(0, min(width - 1, x1))
    x2 = max(0, min(width, x2))
    y1 = max(0, min(height - 1, y1))
    y2 = max(0, min(height, y2))

    if x2 <= x1:
        x2 = min(width, x1 + min_size)
    if y2 <= y1:
        y2 = min(height, y1 + min_size)

    return [x1, y1, x2, y2]


def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = clip_bbox(bbox, w, h)

    bw = x2 - x1
    bh = y2 - y1
    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    border_mask = expanded_mask - face_mask
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox": (x1, y1, x2, y2),
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


def create_face_subregion_masks(frame_shape, bbox):
    h, w = frame_shape[:2]
    x1, y1, x2, y2 = clip_bbox(bbox, w, h)
    cx = (x1 + x2) // 2
    cy = (y1 + y2) // 2

    masks = {}
    for name, coords in {
        "left": (x1, y1, cx, y2),
        "right": (cx, y1, x2, y2),
        "top": (x1, y1, x2, cy),
        "bottom": (x1, cy, x2, y2),
        "tl": (x1, y1, cx, cy),
        "tr": (cx, y1, x2, cy),
        "bl": (x1, cy, cx, y2),
        "br": (cx, cy, x2, y2),
    }.items():
        mask = np.zeros((h, w), dtype=np.uint8)
        sx1, sy1, sx2, sy2 = coords
        if sx2 > sx1 and sy2 > sy1:
            mask[sy1:sy2, sx1:sx2] = 1
        masks[name] = mask

    return masks

## Funcoes de visualizacao

In [ ]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img


def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()
    overlay[regions["face"] == 1] = (0, 255, 0)
    overlay[regions["border"] == 1] = (0, 0, 255)
    overlay[regions["background"] == 1] = (255, 0, 0)
    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)


def illumination_visual(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l_channel = lab[:, :, 0]
    illumination = cv2.GaussianBlur(l_channel, (0, 0), sigmaX=15, sigmaY=15)
    illum_vis = cv2.normalize(illumination, None, 0, 255, cv2.NORM_MINMAX)
    return illum_vis.astype(np.uint8)

# Grupo E - Fisica de iluminacao

In [ ]:
def region_values(channel, mask):
    values = channel[mask == 1]
    if len(values) < 25:
        return None
    return values.astype(np.float32)


def histogram_entropy(values, bins=64, value_range=(0, 255)):
    hist, _ = np.histogram(values, bins=bins, range=value_range, density=False)
    hist = hist.astype(np.float64)
    hist /= (hist.sum() + 1e-12)
    return float(-np.sum(hist * np.log(hist + 1e-12)) / np.log(bins))


def compute_illumination_region(frame_lab, hsv, illumination, grad_mag, grad_angle, mask):
    l_values = region_values(frame_lab[:, :, 0], mask)
    if l_values is None:
        return None

    a_values = region_values(frame_lab[:, :, 1], mask)
    b_values = region_values(frame_lab[:, :, 2], mask)
    s_values = region_values(hsv[:, :, 1], mask)
    v_values = region_values(hsv[:, :, 2], mask)
    illum_values = region_values(illumination, mask)
    grad_values = region_values(grad_mag, mask)
    angle_values = region_values(grad_angle, mask)

    l_mean = float(np.mean(l_values))
    l_std = float(np.std(l_values))
    l_p5 = float(np.percentile(l_values, 5))
    l_p95 = float(np.percentile(l_values, 95))
    l_range = l_p95 - l_p5

    highlight_threshold = max(220.0, l_p95)
    shadow_threshold = min(35.0, l_p5)

    highlight_ratio = float(np.mean(l_values >= highlight_threshold))
    shadow_ratio = float(np.mean(l_values <= shadow_threshold))

    grad_energy = float(np.mean(grad_values)) if grad_values is not None else 0.0
    grad_std = float(np.std(grad_values)) if grad_values is not None else 0.0

    if angle_values is not None and grad_values is not None and np.sum(grad_values) > 1e-6:
        weights = grad_values / (np.sum(grad_values) + 1e-12)
        mean_sin = float(np.sum(weights * np.sin(angle_values)))
        mean_cos = float(np.sum(weights * np.cos(angle_values)))
        grad_coherence = float(np.sqrt(mean_sin ** 2 + mean_cos ** 2))
        grad_direction = float(np.arctan2(mean_sin, mean_cos))
    else:
        grad_coherence = 0.0
        grad_direction = 0.0

    return {
        "l_mean": l_mean,
        "l_std": l_std,
        "l_contrast": float(l_std / (l_mean + 1e-6)),
        "l_range": float(l_range),
        "l_entropy_norm": histogram_entropy(l_values),
        "highlight_ratio": highlight_ratio,
        "shadow_ratio": shadow_ratio,
        "a_mean": float(np.mean(a_values)),
        "b_mean": float(np.mean(b_values)),
        "a_std": float(np.std(a_values)),
        "b_std": float(np.std(b_values)),
        "sat_mean": float(np.mean(s_values)),
        "sat_std": float(np.std(s_values)),
        "value_mean": float(np.mean(v_values)),
        "illum_mean": float(np.mean(illum_values)),
        "illum_std": float(np.std(illum_values)),
        "grad_energy": grad_energy,
        "grad_std": grad_std,
        "grad_coherence": grad_coherence,
        "grad_direction": grad_direction,
    }


def illumination_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    comparable_metrics = [
        "l_mean",
        "l_std",
        "l_contrast",
        "l_range",
        "l_entropy_norm",
        "highlight_ratio",
        "shadow_ratio",
        "a_mean",
        "b_mean",
        "sat_mean",
        "illum_mean",
        "illum_std",
        "grad_energy",
        "grad_std",
        "grad_coherence",
    ]

    features = {
        f"{prefix}_phys_{metric}_diff": abs(f1[metric] - f2[metric])
        for metric in comparable_metrics
    }

    direction_diff = abs(np.angle(np.exp(1j * (f1["grad_direction"] - f2["grad_direction"]))))
    features[f"{prefix}_phys_grad_direction_diff"] = float(direction_diff)

    return features

In [ ]:
def compute_face_light_asymmetry(frame_lab, bbox_scaled):
    sub_masks = create_face_subregion_masks(frame_lab.shape, bbox_scaled)
    l_channel = frame_lab[:, :, 0].astype(np.float32)

    means = {}
    for name, mask in sub_masks.items():
        values = region_values(l_channel, mask)
        means[name] = float(np.mean(values)) if values is not None else np.nan

    lr_den = abs(means["left"]) + abs(means["right"]) + 1e-6
    tb_den = abs(means["top"]) + abs(means["bottom"]) + 1e-6

    quadrant_values = np.array([means["tl"], means["tr"], means["bl"], means["br"]], dtype=np.float32)
    quadrant_values = quadrant_values[~np.isnan(quadrant_values)]

    if len(quadrant_values) == 0:
        quadrant_imbalance = np.nan
    else:
        quadrant_imbalance = float(np.std(quadrant_values) / (np.mean(np.abs(quadrant_values)) + 1e-6))

    return {
        "phys_face_lr_luma_diff": float(abs(means["left"] - means["right"]) / lr_den),
        "phys_face_tb_luma_diff": float(abs(means["top"] - means["bottom"]) / tb_den),
        "phys_face_luma_quadrant_imbalance": quadrant_imbalance,
    }


def compute_physics_metrics(frame, bbox):
    frame_std, scale = standardize_frame(frame)
    bbox_scaled = scale_bbox(bbox, scale)
    bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])

    lab = cv2.cvtColor(frame_std, cv2.COLOR_BGR2LAB)
    hsv = cv2.cvtColor(frame_std, cv2.COLOR_BGR2HSV)
    l_channel = lab[:, :, 0]

    illumination = cv2.GaussianBlur(l_channel, (0, 0), sigmaX=15, sigmaY=15).astype(np.float32)
    gx = cv2.Sobel(illumination, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(illumination, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = cv2.magnitude(gx, gy)
    grad_angle = cv2.phase(gx, gy, angleInDegrees=False)

    regions = create_face_regions(frame_std, bbox_scaled)

    face = compute_illumination_region(lab, hsv, illumination, grad_mag, grad_angle, regions["face"])
    border = compute_illumination_region(lab, hsv, illumination, grad_mag, grad_angle, regions["border"])
    bg = compute_illumination_region(lab, hsv, illumination, grad_mag, grad_angle, regions["background"])

    features = {}

    for region_name, region_features in [("face", face), ("border", border), ("background", bg)]:
        if region_features is None:
            continue

        for metric_name, metric_value in region_features.items():
            features[f"phys_{region_name}_{metric_name}"] = metric_value

    features.update(compute_face_light_asymmetry(lab, bbox_scaled))
    features.update(illumination_region_differences(face, bg, "face_bg"))
    features.update(illumination_region_differences(face, border, "face_border"))
    features.update(illumination_region_differences(border, bg, "border_bg"))

    return features, illumination, grad_mag

## Debug por video

In [ ]:
def debug_physics_video(video_path, max_frames=120, interval=60):
    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    debug_frames = []

    print("\nProcessando fisica de iluminacao...")

    for i, frame in enumerate(tqdm.tqdm(frames)):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        frame_std, scale = standardize_frame(frame)
        bbox = metadata[metadata_idx]["bbox"]
        bbox_scaled = scale_bbox(bbox, scale)
        bbox_scaled = clip_bbox(bbox_scaled, frame_std.shape[1], frame_std.shape[0])
        regions = create_face_regions(frame_std, bbox_scaled)

        features, illumination, grad_mag = compute_physics_metrics(frame, bbox)
        illum_vis = illumination_visual(frame_std)
        grad_vis = cv2.normalize(grad_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {frame_idx} | Meta: {metadata_idx}",
            f"Face L Mean: {features.get('phys_face_l_mean', 0):.2f}",
            f"Face Contrast: {features.get('phys_face_l_contrast', 0):.3f}",
            f"LR Diff: {features.get('phys_face_lr_luma_diff', 0):.3f}",
            f"Face-BG L Diff: {features.get('face_bg_phys_l_mean_diff', 0):.2f}",
            f"Face-Border GradDiff: {features.get('face_border_phys_grad_energy_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(illum_vis, cv2.COLOR_GRAY2RGB),
            cv2.cvtColor(grad_vis, cv2.COLOR_GRAY2RGB),
        ])

        debug_frames.append(combined)

    if not debug_frames:
        raise ValueError("Nenhum frame válido para visualização")

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.axis("off")
    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(fig, update, frames=len(debug_frames), interval=interval, blit=True)

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

# Todas as metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames, frame_indices, frame_count = load_video_frames(video_path, max_frames=max_frames, return_indices=True)
    metadata = load_metadata(video_path)
    all_features = []

    for i, frame in enumerate(frames):
        frame_idx = int(frame_indices[i])
        metadata_idx = metadata_index_for_frame(frame_idx, frame_count, len(metadata))

        if metadata_idx is None:
            continue

        bbox = metadata[metadata_idx]["bbox"]
        features_phys, _, _ = compute_physics_metrics(frame, bbox)

        features = {**features_phys}
        features["frame"] = frame_idx
        features["metadata_idx"] = metadata_idx

        all_features.append(features)

    metrics_df = pd.DataFrame(all_features)
    metrics_df.insert(0, "video_name", video_name)

    if label is not None:
        metrics_df.insert(1, "label", label)

    return metrics_df


def metadata_exists(video_path):
    return os.path.exists(metadata_path_for_video(video_path))


def aggregate_video_metrics(frame_metrics):
    metric_cols = [
        col for col in frame_metrics.columns
        if (col.startswith("phys_") or "_phys_" in col)
    ]

    agg = frame_metrics[metric_cols].agg(["mean", "std", "median"])
    values = {}

    for metric in metric_cols:
        values[f"{metric}_mean"] = agg.loc["mean", metric]
        values[f"{metric}_std"] = agg.loc["std", metric]
        values[f"{metric}_median"] = agg.loc["median", metric]

    return values


def auc_from_scores(labels, scores, positive_label="Fake"):
    valid = [(label, score) for label, score in zip(labels, scores) if pd.notna(score)]
    pos = [score for label, score in valid if label == positive_label]
    neg = [score for label, score in valid if label != positive_label]

    if len(pos) == 0 or len(neg) == 0:
        return np.nan

    wins = 0
    ties = 0

    for pos_score in pos:
        for neg_score in neg:
            wins += pos_score > neg_score
            ties += pos_score == neg_score

    return (wins + 0.5 * ties) / (len(pos) * len(neg))


def auc_abs_bootstrap_ci(labels, scores, positive_label="Fake", n_bootstrap=1000, ci=0.95, random_state=42):
    values = pd.DataFrame({"label": labels, "score": scores}).dropna()
    pos = values.loc[values["label"] == positive_label, "score"].astype(float).to_numpy()
    neg = values.loc[values["label"] != positive_label, "score"].astype(float).to_numpy()

    if len(pos) < 2 or len(neg) < 2:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(random_state)
    boot = []

    for _ in range(n_bootstrap):
        pos_sample = rng.choice(pos, size=len(pos), replace=True)
        neg_sample = rng.choice(neg, size=len(neg), replace=True)
        sample_labels = [positive_label] * len(pos_sample) + ["__negative__"] * len(neg_sample)
        sample_scores = np.concatenate([pos_sample, neg_sample])
        auc = auc_from_scores(sample_labels, sample_scores, positive_label=positive_label)
        if pd.notna(auc):
            boot.append(max(auc, 1 - auc))

    if not boot:
        return np.nan, np.nan, np.nan

    alpha = (1 - ci) / 2
    return (
        float(np.quantile(boot, alpha)),
        float(np.quantile(boot, 1 - alpha)),
        float(np.std(boot, ddof=1)) if len(boot) > 1 else 0.0,
    )


def cohen_d_by_label(values, labels, positive_label="Fake"):
    values = pd.Series(values)
    labels = pd.Series(labels)

    pos = values[labels == positive_label].dropna().astype(float)
    neg = values[labels != positive_label].dropna().astype(float)

    if len(pos) < 2 or len(neg) < 2:
        return np.nan

    pooled = np.sqrt((pos.var(ddof=1) + neg.var(ddof=1)) / 2)

    if pooled == 0 or pd.isna(pooled):
        return np.nan

    return (pos.mean() - neg.mean()) / pooled


def classify_metric_scope(metric):
    if "face_bg_" in metric:
        return "face_background_diff"
    if "border_bg_" in metric:
        return "border_background_diff"
    if "face_border_" in metric:
        return "face_border_diff"
    if "_background_" in metric:
        return "background_direct"
    if "_border_" in metric:
        return "border_direct"
    if "_face_" in metric:
        return "face_direct"
    return "other"


def is_context_sensitive_metric(metric):
    scope = classify_metric_scope(metric)
    return scope in {"background_direct", "face_background_diff", "border_background_diff"} or metric.endswith("patch_count_mean")


def build_metric_report(video_metrics, metric_cols, n_bootstrap=1000, random_state=42):
    report_rows = []

    for metric in metric_cols:
        valid = video_metrics[["label", metric]].dropna()
        auc = auc_from_scores(video_metrics["label"], video_metrics[metric])
        auc_abs = max(auc, 1 - auc) if pd.notna(auc) else np.nan
        ci_low, ci_high, auc_bootstrap_std = auc_abs_bootstrap_ci(
            video_metrics["label"],
            video_metrics[metric],
            n_bootstrap=n_bootstrap,
            random_state=random_state,
        )
        d = cohen_d_by_label(video_metrics[metric], video_metrics["label"])

        report_rows.append({
            "metric": metric,
            "scope": classify_metric_scope(metric),
            "context_sensitive": is_context_sensitive_metric(metric),
            "direction": "higher_fake" if pd.notna(auc) and auc >= 0.5 else "higher_real" if pd.notna(auc) else "undefined",
            "auc_fake": auc,
            "auc_abs": auc_abs,
            "auc_abs_ci_low": ci_low,
            "auc_abs_ci_high": ci_high,
            "auc_abs_bootstrap_std": auc_bootstrap_std,
            "cohen_d_fake_minus_real": d,
            "n_fake": int((valid["label"] == "Fake").sum()),
            "n_real": int((valid["label"] == "Real").sum()),
            "missing_rate": float(video_metrics[metric].isna().mean()),
            "real_mean": video_metrics.loc[video_metrics["label"] == "Real", metric].mean(),
            "fake_mean": video_metrics.loc[video_metrics["label"] == "Fake", metric].mean(),
        })

    report = pd.DataFrame(report_rows)

    if not report.empty:
        report = report.sort_values(
            ["auc_abs", "cohen_d_fake_minus_real"],
            ascending=[False, False],
            key=lambda s: s.abs() if s.name == "cohen_d_fake_minus_real" else s,
        ).reset_index(drop=True)

    return report


def evaluate_group_e(max_frames=120):
    rows = []
    available_df = df[df['video_path'].apply(metadata_exists)].reset_index(drop=True)

    for _, row in available_df.iterrows():
        frame_metrics = all_metrics(
            row['video_path'],
            max_frames=max_frames,
            label=row['Video Ground Truth'],
        )

        if frame_metrics.empty:
            continue

        video_row = aggregate_video_metrics(frame_metrics)
        video_row["video_name"] = os.path.basename(row['video_path'])
        video_row["label"] = row['Video Ground Truth']
        video_row["n_frames"] = len(frame_metrics)
        rows.append(video_row)

    video_metrics = pd.DataFrame(rows)

    if video_metrics.empty:
        return video_metrics, pd.DataFrame()

    metric_cols = [
        col for col in video_metrics.columns
        if (col.startswith("phys_") or "_phys_" in col) and col.endswith("_mean")
    ]

    report = build_metric_report(video_metrics, metric_cols)

    print("\nVideos avaliados:")
    print(video_metrics['label'].value_counts())
    print("\nObservacao metodologica: auc_abs ranqueia separabilidade univariada; use direction e IC bootstrap antes de interpretar como sinal robusto.")

    return video_metrics, report


## Avaliacao discriminativa

Executa a extracao apenas nos videos com metadata disponivel e resume uma linha por video. Use `report.head(20)` para ver quais sinais fisicos de iluminacao separam melhor Real/Fake nesta amostra.

In [ ]:
# Rodada rapida de validacao. Aumente max_frames para estabilizar os resultados.
video_metrics, report = evaluate_group_e(max_frames=120)
display(report.head(20))